# Pandas Method Chaining and Data Cleaning

## Real Estate Data Preparation with Multiple CSV Files

This notebook preserves the main concepts from the provided notebook and expands them into an easy-to-teach, real-world data-cleaning workflow.

## Table of Contents

1. [Learning Objectives](#learning-objectives)
2. [Dataset Overview](#dataset-overview)
3. [Import Pandas](#import-pandas)
4. [Define File Paths](#define-file-paths)
5. [Load the CSV Files](#load-the-csv-files)
6. [Inspect Each Dataset](#inspect-each-dataset)
7. [Understand the Schema](#understand-the-schema)
8. [What Is Method Chaining](#what-is-method-chaining)
9. [Clean Dataset 2 with Method Chaining](#clean-dataset-2-with-method-chaining)
10. [Parse Latitude and Longitude](#parse-latitude-and-longitude)
11. [Parse State from Location Text](#parse-state-from-location-text)
12. [Clean Dataset 3](#clean-dataset-3)
13. [Create a Reusable Cleaning Function](#create-a-reusable-cleaning-function)
14. [Validate the Cleaned Data](#validate-the-cleaned-data)
15. [Compare Columns Before Combining](#compare-columns-before-combining)
16. [Combine DataFrames](#combine-dataframes)
17. [Check Missing Values](#check-missing-values)
18. [Important Data Quality Issue](#important-data-quality-issue)
19. [Recommended Final Standardization](#recommended-final-standardization)
20. [Key Concepts](#key-concepts)
21. [Practice Activities](#practice-activities)

## Learning Objectives

Students will learn to:

- Load CSV files with Pandas.
- Inspect DataFrames using `head()`, `shape`, `info()`, `dtypes`, and `isnull().sum()`.
- Understand schema differences between datasets.
- Explain and use method chaining.
- Use `assign()`, `lambda`, `drop()`, `dropna()`, and `.str.split()`.
- Convert text values into numeric values.
- Create reusable cleaning functions.
- Validate schemas before combining datasets.
- Combine DataFrames with `pd.concat()`.
- Identify important data-quality problems before analysis.

## Dataset Overview

The notebook uses three real-estate files:

- `mexico-real-estate-1.csv`
- `mexico-real-estate-2.csv`
- `mexico-real-estate-3.csv`

Important fields include property type, state, latitude, longitude, area, and price.

The files are not perfectly consistent. Dataset 2 uses `price_mxn`, while Dataset 3 stores location and coordinates in combined text columns.

This is a common real-world situation: data comes from different sources and must be standardized before analysis.

## Import Pandas

In [ ]:
# Pandas is used for working with tabular data.
import pandas as pd

### What is Pandas?

Pandas is a Python library for data manipulation and analysis.

Its most important structure is the DataFrame, which works like a programmable table with rows and columns.

## Define File Paths

In [ ]:
# Paths to the three CSV files.
path_1 = "./data/mexico-real-estate-1.csv"
path_2 = "./data/mexico-real-estate-2.csv"
path_3 = "./data/mexico-real-estate-3.csv"

### Why store paths in variables?

It avoids repeating long file paths and makes the notebook easier to maintain.

## Load the CSV Files

In [ ]:
# Load the three CSV files into separate DataFrames.
df1 = pd.read_csv(path_1)
df2 = pd.read_csv(path_2)
df3 = pd.read_csv(path_3)

### `pd.read_csv()`

`pd.read_csv()` reads a CSV file and returns a Pandas DataFrame.

Basic pattern:

```python
df = pd.read_csv("file.csv")
```

After loading data, always inspect it before cleaning or analysis.

## Inspect Each Dataset

In [ ]:
# Preview Dataset 1.
df1.head()

In [ ]:
# Preview Dataset 2.
df2.head()

In [ ]:
# Preview Dataset 3.
df3.head()

### Why use `head()`?

`head()` gives a quick view of the first five rows by default.

It helps us understand column names, values, missing data, and formatting problems.

Use `df.head(10)` when you want to see the first 10 rows.

## Understand the Schema

In [ ]:
# Compare the columns in all three datasets.
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("df3 columns:", df3.columns.tolist())

### What is a schema?

A schema describes the structure of a dataset, including column names and the expected type or meaning of values.

Here we can see important differences:

- Dataset 2 has `price_mxn` instead of `price_usd`.
- Dataset 3 has `lat-lon` instead of separate `lat` and `lon`.
- Dataset 3 has `place_with_parent_names` instead of `state`.

Before combining datasets, these differences should be understood and standardized.

## What Is Method Chaining?

Method chaining means applying several Pandas operations one after another.

Instead of:

```python
df = pd.read_csv(path)
df = df.dropna()
df = df.drop(columns=["old_column"])
```

we can write:

```python
df = (
    pd.read_csv(path)
    .dropna()
    .drop(columns=["old_column"])
)
```

### Why use method chaining?

- Less repetitive code.
- Clear transformation flow.
- Easier to read when transformations are logically connected.
- Useful for building data-cleaning pipelines.

Important: method chaining is a coding style. Students should still understand every method being chained.

## Clean Dataset 2 with Method Chaining

In [ ]:
df2 = (
    pd.read_csv(path_2)

    # Convert Mexican pesos to US dollars.
    # The original notebook uses 19 pesos = 1 USD for the 2014 data.
    .assign(price_usd=lambda x: x["price_mxn"].div(19))

    # Remove the original price_mxn column.
    .drop(columns=["price_mxn"])

    # Remove rows containing missing values.
    .dropna()
)

### Step-by-step

1. `pd.read_csv(path_2)` loads the file.
2. `assign()` creates `price_usd`.
3. `lambda x:` means `x` represents the DataFrame at that point in the chain.
4. `.div(19)` divides each price by 19.
5. `drop()` removes the old currency column.
6. `dropna()` removes rows containing missing values.

`x["price_mxn"].div(19)` is equivalent to:

```python
x["price_mxn"] / 19
```

In [ ]:
# Verify the cleaned Dataset 2.
print("df2 type:", type(df2))
print("df2 shape:", df2.shape)
df2.head()

## Parse Latitude and Longitude

Dataset 3 stores coordinates together:

```text
19.52589,-99.151703
```

For analysis, it is better to have separate `lat` and `lon` columns.

In [ ]:
# Extract latitude from the combined lat-lon string.
(
    pd.read_csv(path_3)["lat-lon"]
    .str.split(",", expand=True)[0]
    .astype(float)
    .head()
)

In [ ]:
# Extract longitude from the combined lat-lon string.
(
    pd.read_csv(path_3)["lat-lon"]
    .str.split(",", expand=True)[1]
    .astype(float)
    .head()
)

### Understanding `.str.split()`

```python
.str.split(",", expand=True)
```

splits each text value at the comma.

`expand=True` creates separate columns.

Then:

- `[0]` selects latitude.
- `[1]` selects longitude.
- `.astype(float)` converts text numbers into numeric values.

Always inspect the structure before extracting values by position.

## Parse State from Location Text

In [ ]:
# Inspect the location hierarchy after splitting it.
(
    pd.read_csv(path_3)["place_with_parent_names"]
    .str.split("|", expand=True)
    .head()
)

In [ ]:
# Extract the state from position 2.
(
    pd.read_csv(path_3)["place_with_parent_names"]
    .str.split("|", expand=True)[2]
    .head()
)

### Why `[2]`?

The location follows a hierarchy similar to:

```text
|México|Estado de México|Toluca|Metepec|
```

Python uses zero-based indexing, so position `[2]` is the third item.

The important lesson is to inspect the split result first rather than guessing the index.

## Clean Dataset 3

In [ ]:
df3 = (
    pd.read_csv(path_3)

    # Remove rows containing missing values.
    .dropna()

    # Create latitude, longitude, and state columns.
    .assign(
        lat=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[0]
            .astype(float),

        lon=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[1]
            .astype(float),

        state=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[2]
    )

    # Remove the original combined columns.
    .drop(columns=["place_with_parent_names", "lat-lon"])
)

In [ ]:
# Verify the cleaned Dataset 3.
print("df3 type:", type(df3))
print("df3 shape:", df3.shape)
df3.head()

Dataset 3 is now easier to analyze because the combined coordinate and location fields have been converted into separate analytical columns.

## Create a Reusable Cleaning Function

In [ ]:
def clean_dataset(path):
    'Load a CSV file and remove rows containing missing values.'
    return (
        pd.read_csv(path)
        .dropna()
    )

### Why use a function?

A function lets us reuse the same logic without rewriting it.

For example:

```python
data = clean_dataset(path_1)
```

Functions improve consistency and reduce repeated code.

Important: this simple function only performs the common cleaning step. Dataset-specific transformations still need their own logic.

In [ ]:
# Use the reusable function on Dataset 1.
data = clean_dataset(path_1)
data.head()

## Validate the Cleaned Data

In [ ]:
# Check the number of rows and columns.
data.shape

### Understanding `shape`

`shape` returns:

```text
(rows, columns)
```

For example, `(583, 6)` means 583 rows and 6 columns.

In [ ]:
# Detailed DataFrame summary.
data.info()

### Understanding `info()`

`info()` shows column names, non-null counts, data types, and memory information.

It is one of the most useful first checks after loading or cleaning data.

In [ ]:
# Show the data type of every column.
data.dtypes

### Why are data types important?

A price stored as text cannot be safely used for mathematical calculations until it is converted to a numeric type.

Common types include:

- `int64` for integers.
- `float64` for decimal numbers.
- `str` or `object` for text.
- `bool` for True/False values.

In [ ]:
# Count missing values in each column.
data.isnull().sum()

### Understanding `isnull().sum()`

`isnull()` identifies missing values.

`sum()` counts those missing values column by column.

This is a simple but important data-quality check.

## Compare Columns Before Combining

In [ ]:
# Check whether all three DataFrames have the same column names.
columns_match = set(df1.columns) == set(df2.columns) == set(df3.columns)

if columns_match:
    print("All DataFrames have the same columns.")
else:
    print("Column mismatch detected!")
    print("df1:", df1.columns.tolist())
    print("df2:", df2.columns.tolist())
    print("df3:", df3.columns.tolist())

### Important limitation

A set comparison checks column names only.

It does not check:

- Column order.
- Data types.
- Units.
- Value formats.
- Meaning of each column.

Professional data preparation checks all of these before combining datasets.

## Combine DataFrames

In [ ]:
# Stack the three DataFrames vertically.
df = pd.concat([df1, df2, df3], ignore_index=True)

print("Combined shape:", df.shape)
df.head()

### Understanding `pd.concat()`

`pd.concat()` combines Pandas objects.

For vertical concatenation:

```python
pd.concat([df1, df2, df3], ignore_index=True)
```

puts the rows of the DataFrames one after another.

`ignore_index=True` creates a new continuous index.

### `concat()` versus `merge()`

- `concat()` is commonly used to stack similar datasets.
- `merge()` is used to join datasets using matching keys.

## Check Missing Values

In [ ]:
# Check missing values after concatenation.
df.isnull().sum()

Always validate after a major transformation.

Combining data can expose inconsistent schemas, missing values, data-type differences, and unexpected columns.

## Important Data Quality Issue

The original notebook contains an important issue worth teaching.

Dataset 1 displays prices such as:

```text
$67,965.56
```

This indicates that `price_usd` may be stored as text.

Dataset 2 converts its price to numeric values, while Dataset 3 also produces numeric `price_usd`.

Therefore, simply concatenating the original DataFrames can result in inconsistent data types.

### Why does this matter?

A numeric price column is needed for:

- Mean price.
- Minimum and maximum price.
- Price per square meter.
- Correlation analysis.
- Visualization.
- Machine learning.

This is why schema standardization should happen before final analysis.

## Recommended Final Standardization

In [ ]:
# Clean Dataset 1 and convert currency text into numeric values.
df1_clean = (
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd=lambda x: (
            x["price_usd"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
    )
)

In [ ]:
# Clean Dataset 2 and standardize its price column.
df2_clean = (
    pd.read_csv(path_2)
    .dropna()
    .assign(price_usd=lambda x: x["price_mxn"].div(19))
    .drop(columns=["price_mxn"])
)

In [ ]:
# Clean Dataset 3 and standardize its location columns.
df3_clean = (
    pd.read_csv(path_3)
    .dropna()
    .assign(
        lat=lambda x: x["lat-lon"].str.split(",", expand=True)[0].astype(float),
        lon=lambda x: x["lat-lon"].str.split(",", expand=True)[1].astype(float),
        state=lambda x: x["place_with_parent_names"].str.split("|", expand=True)[2]
    )
    .drop(columns=["place_with_parent_names", "lat-lon"])
)

In [ ]:
# Check that the standardized DataFrames use the same column names.
print("df1_clean:", df1_clean.columns.tolist())
print("df2_clean:", df2_clean.columns.tolist())
print("df3_clean:", df3_clean.columns.tolist())

print(
    "Same columns:",
    set(df1_clean.columns) == set(df2_clean.columns) == set(df3_clean.columns)
)

In [ ]:
# Check data types before combining.
print("df1_clean dtypes:")
print(df1_clean.dtypes)

print("\ndf2_clean dtypes:")
print(df2_clean.dtypes)

print("\ndf3_clean dtypes:")
print(df3_clean.dtypes)

### Why this is better

Before final concatenation, we have standardized:

- Column names.
- Coordinate columns.
- Price representation.
- Data types.
- Missing-value handling.

This is safer for downstream analysis.

## Combine the Standardized Data

In [ ]:
# Combine only after standardization.
df_final = pd.concat(
    [df1_clean, df2_clean, df3_clean],
    ignore_index=True
)

print("Final shape:", df_final.shape)
df_final.head()

In [ ]:
# Final validation.
print("Missing values:")
print(df_final.isnull().sum())

print("\nData types:")
print(df_final.dtypes)

print("\nShape:")
print(df_final.shape)

## Key Concepts

### 1. DataFrame
A table-like Pandas structure containing rows and columns.

### 2. `read_csv()`
Loads CSV data into a DataFrame.

### 3. `head()`
Quickly previews rows.

### 4. `shape`
Returns `(rows, columns)`.

### 5. `info()`
Shows structure, non-null counts, and data types.

### 6. `dtypes`
Shows the type of every column.

### 7. `isnull().sum()`
Counts missing values.

### 8. Method chaining
Applies multiple transformations in one expression.

### 9. `assign()`
Creates or modifies columns.

### 10. `lambda`
Defines a short function, commonly used inside method chains.

### 11. `.str.split()`
Splits text into parts.

### 12. `astype(float)`
Converts values to decimal numbers.

### 13. `dropna()`
Removes rows containing missing values.

### 14. `pd.concat()`
Combines DataFrames, commonly by stacking rows.

### 15. Schema standardization
Makes datasets structurally and semantically compatible before combining them.

## Practice Activities

### Activity 1: Inspect the Data

Run:

```python
df1.shape
df1.info()
df1.dtypes
df1.isnull().sum()
```

Explain what each output tells you.

### Activity 2: Method Chaining

Create a chain that loads Dataset 1, removes missing rows, and selects required columns.

### Activity 3: Text Splitting

Extract latitude and longitude from `lat-lon`.

### Activity 4: Currency Cleaning

Convert:

```text
$67,965.56
```

into:

```text
67965.56
```

Then explain why numeric conversion is necessary.

### Activity 5: Schema Validation

Check whether the cleaned DataFrames have the same columns and compatible data types.

### Activity 6: Think Like a Data Scientist

If two datasets have the same column names but different units, can they safely be combined?

**Expected answer:** No. Column names alone are not enough. Units, meaning, formats, and data types must also be compatible.

## Final Data Science Takeaway

A reliable data-preparation workflow is:

**Load -> Inspect -> Understand -> Clean -> Transform -> Standardize -> Validate -> Combine -> Validate Again -> Analyze**

Good analysis depends on good data preparation.

Always check the structure, quality, data types, units, and meaning of your data before performing calculations, visualization, or machine learning.